In [ ]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.5 MB/s eta 0:00:00:00:0100:01


FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
try: 
    train = pd.read_csv('train.csv') # colab
else: 
    train = pd.read_csv('../data/train.csv') # local

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [6]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
# q1
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

result_q1 = zs(prompt_150, labels_150)
idx_q1 = result_q1['labels'].index(ans_150)
print(round(result_q1['scores'][idx_q1], 3))

0.384


In [8]:
# q2
prompt_150_emb = model.encode([prompt_150])
distances_q2, indices_q2 = index.search(prompt_150_emb, 10)
rank_q2 = list(indices_q2[0]).index(150) + 1
print(rank_q2)

10


In [10]:
# q3
retrieved_indices = indices_q2[0]
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

true_doc = kb[150]
true_pos = docs_10.index(true_doc)
sorted_positions = np.argsort(ce_scores)[::-1]
rank_ce = list(sorted_positions).index(true_pos) + 1
print(rank_ce)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

1


In [11]:
# q4
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
_, indices_42 = index.search(model.encode([prompt_42]), 5)
context_str = " ".join([kb[i] for i in indices_42[0]])
rag_str_q4 = f"Context: {context_str} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens_q4 = tokenizer(rag_str_q4)
print(len(tokens_q4['input_ids']))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

216


In [12]:
# q5
rag_str_q5 = f"Context: {true_doc} Question: {prompt_150}"
result_q5 = zs(rag_str_q5, labels_150)
idx_q5 = result_q5['labels'].index(ans_150)
print(round(result_q5['scores'][idx_q5], 3))

0.989


In [13]:
# q6
wrong_doc = kb[999]
rag_str_adv = f"Context: {wrong_doc} Question: {prompt_150}"
result_q6 = zs(rag_str_adv, labels_150)
idx_q6 = result_q6['labels'].index(ans_150)
print(round(result_q6['scores'][idx_q6], 3))

0.529


In [14]:
# q7
hits = 0
for i in range(100):
    row_i = train.iloc[i]
    p_i = str(row_i['prompt'])
    correct_opt_i = str(row_i[row_i['answer']])
    _, idxs_i = index.search(model.encode([p_i]), 5)
    retrieved_docs_i = [kb[j] for j in idxs_i[0]]
    if any(correct_opt_i in doc for doc in retrieved_docs_i):
        hits += 1
print(float(hits))

73.0


In [15]:
# q8
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score

ap_scores = []
for i in range(20):
    row_i = train.iloc[i]
    p_i = str(row_i['prompt'])
    labels_i = [str(row_i['A']), str(row_i['B']), str(row_i['C']), str(row_i['D']), str(row_i['E'])]
    ans_i = str(row_i[row_i['answer']])

    p_emb_i = model.encode([p_i])
    _, idxs_i = index.search(p_emb_i, 5)
    docs_5_i = [kb[j] for j in idxs_i[0]]

    pairs_i = [[p_i, doc] for doc in docs_5_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5_i[np.argmax(ce_scores_i)]

    rag_str_i = f"Context: {best_doc} Question: {p_i}"
    result_i = zs(rag_str_i, labels_i)

    sorted_labels = [x for _, x in sorted(zip(result_i['scores'], result_i['labels']), reverse=True)]
    ap_scores.append(apk(ans_i, sorted_labels, k=3))

print(round(np.mean(ap_scores), 3))

0.975
